# JetClass unique-pass scaling-law plot

Grow this plot as unique-pass runs land. **No training in this notebook.**

A point is one unique pass: $n = S \times B$ distinct jets, $B = 512 \times n_\mathrm{GPU}$, val CE $\ell$ at that $S$. Compute on the plot is

$$C(S) = (\text{fwd+bwd FLOPs per optimizer step}) \times S$$

from `cpen.training.scaling_compute` (net + incidence, **not** star-$R$ construction).

Paused s25k jobs ($L=3,D=256$ and later $L=5,D=512$) are **constant-$P$ slices** of $\ell(n)$, not a width or depth family. They can sit on the plot once $\eta_0^\star$ is chosen; they cannot define iso-$C$ troughs by themselves.

In [ ]:
from __future__ import annotations

import json
import os
import re
import sys
from pathlib import Path

%matplotlib inline
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.pyplot as plt
import pandas as pd

REPO = Path("../..").resolve()
sys.path.insert(0, str(REPO / "src"))

from cpen.utils.transfer_plots import JETCLASS_OUT_ROOT, savefig, setup_mpl_style

setup_mpl_style()
FIG_DIR = REPO / "notebooks" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
FLOP_CACHE = FIG_DIR / "scaling_flops.json"
OUT_ROOT = JETCLASS_OUT_ROOT
SLURM_LOG_ROOT = Path("/scratch/gpfs/BHANIN/jdezoort/cpen_runs/jetclass_runs")
GLOBAL_BATCH = 512 * 4
print("OUT_ROOT", OUT_ROOT)
print("FLOP_CACHE", FLOP_CACHE)

## FLOPs per optimizer step

Measure once per $(L,D,H)$. Re-run the cell with `FORCE=True` to recompute. Construction often counts as 0 GEMM FLOPs; leave it out of $C$.

In [ ]:
from cpen.training.scaling_compute import measure_flops

FORCE = False
ARCHS = [(3, 256, 8), (5, 512, 16)]
cache = json.loads(FLOP_CACHE.read_text()) if FLOP_CACHE.is_file() else {}

for L, D, H in ARCHS:
    key = f"L{L}_D{D}_H{H}"
    if key in cache and not FORCE:
        print("cached", key, "flops/step", cache[key]["flops_per_optimizer_step"])
        continue
    print("measuring", key, "…")
    report = measure_flops(
        depth=L, width=D, heads=H,
        per_gpu_batch=2, n_gpus=4, num_particles=80, device="cpu",
    )
    rec = {
        "n_params": report.n_params,
        "flops_per_optimizer_step": report.flops_per_optimizer_step,
        "flops_train_per_gpu": report.flops_train_per_gpu,
        "flops_graph_per_gpu": report.flops_graph_per_gpu,
        "kaplan_6Pnp_per_jet": report.kaplan_6Pnp_per_jet,
        "per_gpu_batch_measured": report.per_gpu_batch,
    }
    # Scale GEMMs linearly in per-GPU batch 2 → 512
    scale = 512 / report.per_gpu_batch
    rec["flops_per_optimizer_step"] *= scale
    rec["flops_train_per_gpu"] *= scale
    rec["note"] = "scaled linearly 2→512 per GPU; construction not in C"
    cache[key] = rec
    print(key, rec)

FLOP_CACHE.write_text(json.dumps(cache, indent=2))
pd.DataFrame(cache).T

## Unique-pass points from paused s25k logs

Each `[val]` line is $\ell$ after $S$ unique-pass steps, $n = S \times 2048$.

In [ ]:
VAL_RE = re.compile(r"\[val\] epoch=(?P<epoch>\d+) step=(?P<step>\d+)(?P<rest>.*)")
KV_RE = re.compile(r"(val_\w+)=([-+eE0-9.]+)")
HEADER_RE = re.compile(r"eta0=(?P<eta0>\S+)\s+arch=L(?P<L>\d+)_D(?P<D>\d+)_h(?P<H>\d+)")
TASK_ETA = {0: 0.05, 1: 0.25, 2: 0.5, 3: 1.0}

rows = []
for path in sorted(SLURM_LOG_ROOT.glob("jc_att*s25k_*.out")):
    ident = {"eta0": None, "depth": None, "width": None, "heads": None}
    mname = re.match(r"(jc_att(?:_L5)?_s25k)_(\d+)_(\d+)\.out$", path.name)
    if mname:
        ident["eta0"] = TASK_ETA.get(int(mname.group(3)))
        ident["depth"], ident["width"], ident["heads"] = (
            (5, 512, 16) if "L5" in mname.group(1) else (3, 256, 8)
        )
    text = path.read_text(errors="replace")
    hm = HEADER_RE.search(text)
    if hm:
        ident.update(eta0=float(hm.group("eta0")), depth=int(hm.group("L")),
                     width=int(hm.group("D")), heads=int(hm.group("H")))
    for vm in VAL_RE.finditer(text):
        row = dict(ident)
        row["S"] = int(vm.group("step"))
        row["n"] = row["S"] * GLOBAL_BATCH
        for k, v in KV_RE.findall(vm.group("rest")):
            row[k] = float(v)
        row["family"] = "mixed"
        rows.append(row)

val = pd.DataFrame(rows)
if not val.empty:
    val = val.drop_duplicates(["depth", "width", "heads", "eta0", "S"], keep="last")
    val["P"] = [
        cache.get(f"L{int(r.depth)}_D{int(r.width)}_H{int(r.heads)}", {}).get("n_params")
        for r in val.itertuples()
    ]
    val["C"] = [
        cache.get(f"L{int(r.depth)}_D{int(r.width)}_H{int(r.heads)}", {}).get("flops_per_optimizer_step", float("nan"))
        * r.S
        for r in val.itertuples()
    ]
print("points", len(val))
val.head(20) if len(val) else val

## $\ell$ vs unique jets (constant $P$ slices)

This is the only panel we can draw until a width or depth family exists.

In [ ]:
if val.empty or "val_loss" not in val.columns:
    print("No unique-pass val_loss yet.")
else:
    fig, ax = plt.subplots(figsize=(4.2, 2.6))
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    i = 0
    for (L, D, H, eta0), sub in val.groupby(["depth", "width", "heads", "eta0"]):
        sub = sub.sort_values("n")
        ax.plot(
            sub["n"], sub["val_loss"], "o-", color=colors[i % len(colors)],
            markersize=3, linewidth=0.9,
            label=rf"$L={int(L)}, D={int(D)}, \eta_0={eta0:g}$",
        )
        i += 1
    ax.set_xscale("log")
    ax.set_xlabel(r"unique jets $n = S \times B$")
    ax.set_ylabel(r"val CE $\ell$")
    ax.grid(True, which="both", alpha=0.25)
    ax.legend(fontsize=6.5, frameon=False)
    fig.tight_layout()
    savefig(fig, "jetclass_scaling_law_ell_vs_n", fig_dir=FIG_DIR)
    plt.show()